# 1. 뉴스기사 요약해보기
새로운 데이터셋에 대해서 추상적 요약과 추출적 요약을 모두 해보는 시간을 가져봐요.

먼저 주요 라이브러리 버전을 확인해 보죠.

!pip install --upgrade summa
!pip install --upgrade nltk #3.9.1

In [1]:
from importlib.metadata import version
import nltk
import torch
import summa
import pandas as pd

print(nltk.__version__)
print(torch.__version__)
print(pd.__version__)
print(version('summa'))

3.9.1
2.12.0+cu132
2.2.2
1.2.0


# Step 1. 데이터 수집하기
데이터는 아래 링크에 있는 뉴스 기사 데이터(news_summary_more.csv)를 사용하세요.

sunnysai12345/News_Summary
아래의 코드로 데이터를 다운로드할 수 있어요.

In [2]:
import urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/sunnysai12345/News_Summary/master/news_summary_more.csv", filename="news_summary_more.csv")
data = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')

In [3]:
data.sample(10)

,headlines,text
36500,KL Rahul smashes fastest-ever IPL 50 off 14 balls,Kings XI Punjab's wicketkeeper-batsman Lokesh ...
20950,Olympic medallist Sushil out of Asian Games af...,India's two-time Olympic medallist wrestler Su...
11452,YSRCP chief moves HC for outside agency probe ...,YSRCP chief Jagan Mohan Reddy has moved the Hy...
38853,"I once told mom, I wish to debut in film like ...",Janhvi Kapoor revealed that once while watchin...
96823,SBI's Arundhati Bhattacharya in greatest leade...,State Bank of India (SBI) Chairperson Arundhat...
54015,Should I catch fish? asks minister seeking ano...,Expressing disappointment over being allotted ...
48060,Sabyasachi slammed for shaming women not knowi...,Designer Sabyasachi Mukherjee has been slammed...
278,Pakistan appoints its first ever female Hindu ...,Suman Kumari has become the first Hindu woman ...
25691,Bottle of whiskey in Venezuela now costs 1 bil...,A litre of top-shelf Scotch whiskey in Venezue...
52106,Scientists developing AI to convert dog bark i...,Scientists in the US are reportedly working to...


이 데이터는 기사의 본문에 해당되는 text와 headlines 두 가지 열로 구성되어져 있습니다.

추상적 요약을 하는 경우에는 text를 본문, headlines를 이미 요약된 데이터로 삼아서 모델을 학습할 수 있어요. 추출적 요약을 하는 경우에는 오직 text열만을 사용하세요.

# Step 2. 데이터 전처리하기 (추상적 요약)
실습에서 사용된 전처리를 참고하여 각자 필요하다고 생각하는 전처리를 추가 사용하여 텍스트를 정규화 또는 정제해 보세요. 만약, 불용어 제거를 선택한다면 상대적으로 길이가 짧은 요약 데이터에 대해서도 불용어를 제거하는 것이 좋을지 고민해 보세요.


In [7]:
import numpy as np
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
from collections import Counter
import torch
from torch.utils.data import TensorDataset, DataLoader

# NLTK 불용어 다운로드
import nltk
nltk.download('stopwords')

# 정규화 사전 정의
contractions = {
    "ain't": "is not", "aren't": "are not","can't": "cannot", "'cause": "because", "could've": "could have", "couldn't": "could not",
    "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not", "haven't": "have not",
    "he'd": "he would","he'll": "he will", "he's": "he is", "how'd": "how did", "how'll": "how will", "how's": "how is",
    "I'd": "I would", "I'll": "I will", "I'm": "I is", "I've": "I have", "isn't": "is not", "it'd": "it would",
    "it'll": "it will", "it's": "it is", "let's": "let us", "ma'am": "madam", "might've": "might have", "mightn't": "might not",
    "must've": "must have", "mustn't": "must not", "needn't": "need not", "oughtn't": "ought not", "shan't": "shall not",
    "sha'n't": "shall not", "should've": "should have", "shouldn't": "should not", "that'd": "that would", "that's": "that is",
    "there'd": "there would", "there's": "there is", "where's": "where is", "they'd": "they would", "they'll": "they will",
    "they're": "they is", "they've": "they have", "we'd": "we would", "we'll": "we will", "we're": "we is", "we've": "we have",
    "weren't": "were not", "what'll": "what will", "what_re": "what is", "what's": "what is", "what_ve": "what have",
    "when's": "when is", "where_d": "where did", "where_ll": "where will", "where_s": "where is", "who'll": "who will",
    "who_s": "who is", "who_ve": "who have", "why's": "why is", "why_re": "why is", "why_ve": "why have", "will've": "will have",
    "won't": "will not", "won't_ve": "will not have", "would've": "would have", "wouldn't": "would not", "wouldn't_ve": "would not have",
    "y'all": "you all", "y'all_d": "you all would", "y'all_re": "you all is", "y'all_ve": "you all have", "you'd": "you would",
    "you'll": "you will", "you're": "you is", "you've": "you have"
}

stop_words = set(stopwords.words('english'))

def preprocess_sentence(sentence, remove_stopwords=True):
    sentence = sentence.lower() # 텍스트 소문자화
    sentence = BeautifulSoup(sentence, "lxml").text # html 태그 제거
    sentence = re.sub(r'\([^)]*\)', '', sentence) # 괄호로 묶인 문자열 제거
    sentence = re.sub('"','', sentence) # 쌍따옴표 제거
    
    # 약어 정규화
    sentence = ' '.join([contractions[t] if t in contractions else t for t in sentence.split(" ")])
    sentence = re.sub(r"'s\b","",sentence) # 소유격 제거
    sentence = re.sub("[^a-zA-Z]", " ", sentence) # 영어 외 문자 공백 변환
    sentence = re.sub('[m]{2,}', 'mm', sentence) # m이 3개 이상이면 2개로 축소
    
    # 불용어 제거 (Text 전처리 시에만 수행)
    if remove_stopwords:
        tokens = ' '.join([word for word in sentence.split() if not word in stop_words if word])
    else:
        tokens = ' '.join([word for word in sentence.split() if word])
    
    return tokens

# 데이터셋의 'text'와 'headlines' 열 전처리 수행
clean_text = [preprocess_sentence(text) for text in data['text']]
clean_headlines = [preprocess_sentence(headline, remove_stopwords=False) for headline in data['headlines']]

data['text'] = clean_text
data['headlines'] = clean_headlines

# 빈 값을 Null로 변환 후 제거
data.replace('', np.nan, inplace=True)
data.dropna(axis=0, inplace=True)

# headlines 앞뒤로 시작 토큰(sostoken)과 종료 토큰(eostoken) 삽입
data['decoder_input'] = data['headlines'].apply(lambda x : 'sostoken ' + x)
data['decoder_target'] = data['headlines'].apply(lambda x : x + ' eostoken')

# 최대 길이 설정 (뉴스 데이터 특성에 맞춰 text 50, headline 12로 조정)
text_max_len = 50
headlines_max_len = 12

def below_threshold_len(max_len, nested_list):
    cnt = 0
    for s in nested_list:
        if(len(s.split()) <= max_len):
            cnt = cnt + 1
    print('전체 샘플 중 길이가 %s 이하인 샘플의 비율: %s'%(max_len, (cnt / len(nested_list))))

below_threshold_len(text_max_len, data['text'])
below_threshold_len(headlines_max_len, data['headlines'])

# 최대 길이 이하인 샘플들만 필터링
data = data[data['text'].apply(lambda x: len(x.split()) <= text_max_len)]
data = data[data['headlines'].apply(lambda x: len(x.split()) <= headlines_max_len)]

# Train / Test 분할을 위한 셔플링
indices = np.arange(data.shape[0])
np.random.shuffle(indices)
data = data.iloc[indices]

n_of_val = int(len(data) * 0.2)
encoder_input = data['text'].values
decoder_input = data['decoder_input'].values
decoder_target = data['decoder_target'].values

encoder_input_train = encoder_input[:-n_of_val]
decoder_input_train = decoder_input[:-n_of_val]
decoder_target_train = decoder_target[:-n_of_val]

encoder_input_test = encoder_input[-n_of_val:]
decoder_input_test = decoder_input[-n_of_val:]
decoder_target_test = decoder_target[-n_of_val:]

# 정수 인코딩 및 토크나이저 구현 함수
def build_tokenizer(texts, vocab_size):
    # <PAD>, <UNK> 외에 SOS, EOS 토큰도 특수 토큰으로 명시하여 딕셔너리에 무조건 포함되도록 합니다.
    special_tokens = {
        "<PAD>": 0, 
        "<UNK>": 1, 
        "sostoken": 2, 
        "eostoken": 3
    }
    
    word_counter = Counter()
    for text in texts:
        word_counter.update(text.split())
        
    # 이미 상단에서 4개의 토큰을 정의했으므로, 빈도수가 높은 단어는 vocab_size - 4 개만큼 추출합니다.
    # 이때 이미 빈도수 계산에 포함되어 있을 'sostoken'과 'eostoken'은 중복 등록되지 않도록 제외합니다.
    idx = 4
    vocab = {}
    for word, _ in word_counter.most_common():
        if len(vocab) >= (vocab_size - 4):
            break
        if word not in special_tokens:
            vocab[word] = idx
            idx += 1
            
    # 특수 토큰들을 단어 사전에 최종 병합합니다.
    vocab.update(special_tokens)
    
    def tokenize(text):
        return [vocab.get(word, vocab["<UNK>"]) for word in text.split()]
        
    return tokenize, vocab

src_vocab_size = 8000
tar_vocab_size = 2000

src_tokenizer, src_vocab = build_tokenizer(encoder_input_train, src_vocab_size)
tar_tokenizer, tar_vocab = build_tokenizer(decoder_input_train, tar_vocab_size)

# 시퀀스 정수 변환 및 패딩 수행
def pad_sequences(sequences, maxlen):
    padded = []
    for seq in sequences:
        if len(seq) < maxlen:
            seq = seq + [0] * (maxlen - len(seq))
        else:
            seq = seq[:maxlen]
        padded.append(seq)
    return torch.tensor(padded, dtype=torch.long)

encoder_input_train = pad_sequences([src_tokenizer(x) for x in encoder_input_train], text_max_len)
decoder_input_train = pad_sequences([tar_tokenizer(x) for x in decoder_input_train], headlines_max_len)
decoder_target_train = pad_sequences([tar_tokenizer(x) for x in decoder_target_train], headlines_max_len)

encoder_input_test = pad_sequences([src_tokenizer(x) for x in encoder_input_test], text_max_len)
decoder_input_test = pad_sequences([tar_tokenizer(x) for x in decoder_input_test], headlines_max_len)
decoder_target_test = pad_sequences([tar_tokenizer(x) for x in decoder_target_test], headlines_max_len)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\H11\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


전체 샘플 중 길이가 50 이하인 샘플의 비율: 1.0
전체 샘플 중 길이가 12 이하인 샘플의 비율: 1.0



# Step 3. 어텐션 메커니즘 사용하기 (추상적 요약)
일반적인 seq2seq보다는 어텐션 메커니즘을 사용한 seq2seq를 사용하는 것이 더 나은 성능을 얻을 수 있어요. 실습 내용을 참고하여 어텐션 메커니즘을 사용한 seq2seq를 설계해 보세요.


In [8]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 인코더 레이어 정의
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        
    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        # 양방향 hidden state 및 cell state 결합 (Concat)
        hidden = torch.cat((hidden[0], hidden[1]), dim=1).unsqueeze(0)
        cell = torch.cat((cell[0], cell[1]), dim=1).unsqueeze(0)
        return outputs, (hidden, cell)

# 바다나우(Bahdanau) 어텐션 레이어 정의
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(BahdanauAttention, self).__init__()
        self.W1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.W2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, query, values):
        # query: (1, batch_size, hidden_dim * 2) -> (batch_size, 1, hidden_dim * 2)
        query = query.transpose(0, 1)
        score = self.V(torch.tanh(self.W1(values) + self.W2(query)))
        attention_weights = F.softmax(score, dim=1)
        context_vector = attention_weights * values
        context_vector = torch.sum(context_vector, dim=1)
        return context_vector, attention_weights

# 디코더 레이어 정의
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim * 2, hidden_dim * 2, batch_first=True)
        self.attention = BahdanauAttention(hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        # x: (batch_size, 1)
        embedded = self.embedding(x)
        context_vector, attention_weights = self.attention(hidden, encoder_outputs)
        
        # 임베딩 벡터와 어텐션 컨텍스트 벡터 결합
        concat_input = torch.cat((embedded, context_vector.unsqueeze(1)), dim=-1)
        output, (hidden, cell) = self.lstm(concat_input, (hidden, cell))
        output = self.fc(output.squeeze(1))
        return output, hidden, cell, attention_weights

# 전체 seq2seq 모델 구성
class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2SeqAttention, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        trg_len = trg.size(1)
        trg_vocab_size = self.decoder.fc.out_features
        
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(device)
        encoder_outputs, (hidden, cell) = self.encoder(src)
        
        decoder_input = trg[:, 0].unsqueeze(1) # sostoken 초기화
        
        for t in range(1, trg_len):
            output, hidden, cell, _ = self.decoder(decoder_input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            top1 = output.argmax(1)
            decoder_input = trg[:, t].unsqueeze(1) if torch.rand(1).item() < teacher_forcing_ratio else top1.unsqueeze(1)
            
        return outputs

# 하이퍼파라미터 세팅 및 모델 선언
embedding_dim = 128
hidden_dim = 128

encoder = Encoder(src_vocab_size, embedding_dim, hidden_dim)
decoder = Decoder(tar_vocab_size, embedding_dim, hidden_dim)
model = Seq2SeqAttention(encoder, decoder).to(device)

# DataLoader 인스턴스화 및 학습 시작
batch_size = 256
train_dataset = TensorDataset(encoder_input_train, decoder_input_train, decoder_target_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

model.train()
for epoch in range(1, 11): # 빠른 프로토타이핑을 위한 10 epoch 설정
    total_loss = 0
    for src_batch, trg_batch, target_batch in train_loader:
        src_batch, trg_batch, target_batch = src_batch.to(device), trg_batch.to(device), target_batch.to(device)
        
        optimizer.zero_grad()
        output = model(src_batch, trg_batch)
        
        loss = criterion(output.view(-1, tar_vocab_size), target_batch.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print(r"Epoch: %d, Avg Loss: %.4f" % (epoch, total_loss / len(train_loader)))

Epoch: 1, Avg Loss: 5.3103
Epoch: 2, Avg Loss: 4.9845
Epoch: 3, Avg Loss: 4.7231
Epoch: 4, Avg Loss: 4.5315
Epoch: 5, Avg Loss: 4.3707
Epoch: 6, Avg Loss: 4.2384
Epoch: 7, Avg Loss: 4.1184
Epoch: 8, Avg Loss: 4.0190
Epoch: 9, Avg Loss: 3.9316
Epoch: 10, Avg Loss: 3.8489



# Step 4. 실제 결과와 요약문 비교하기 (추상적 요약)
원래의 요약문(headlines 열)과 학습을 통해 얻은 추상적 요약의 결과를 비교해 보세요.



In [9]:
# 인덱스를 텍스트로 복원하기 위한 역변환 딕셔너리 생성
src_index_to_word = {v: k for k, v in src_vocab.items()}
tar_index_to_word = {v: k for k, v in tar_vocab.items()}

def seq_to_text(seq):
    return ' '.join([src_index_to_word[idx] for idx in seq if idx not in [0, 1]])

def seq_to_headline(seq):
    return ' '.join([tar_index_to_word[idx] for idx in seq if idx not in [0, 1, tar_vocab['sostoken'], tar_vocab['eostoken']]])

# 예측(Inference) 테스트를 위한 샘플 출력 함수
# Step 4 인퍼런스 루프 내부의 에러 방지 안전 코드 예시
model.eval()
with torch.no_grad():
    for i in range(0, 3):
        src_seq = encoder_input_test[i].unsqueeze(0).to(device)
        
        encoder_outputs, (hidden, cell) = model.encoder(src_seq)
        
        # tar_vocab.get('sostoken', 2) 처럼 기본값을 지정해주면 KeyError를 방지할 수 있습니다.
        sos_idx = tar_vocab.get('sostoken', 2)
        eos_idx = tar_vocab.get('eostoken', 3)
        
        decoder_input = torch.tensor([[sos_idx]]).to(device)
        
        decoded_words = []
        for t in range(headlines_max_len):
            output, hidden, cell, _ = model.decoder(decoder_input, hidden, cell, encoder_outputs)
            top1 = output.argmax(1).item()
            
            if top1 == eos_idx:
                break
            decoded_words.append(top1)
            decoder_input = torch.tensor([[top1]]).to(device)
            
        print("원문 : ", seq_to_text(encoder_input_test[i].numpy()))
        print("실제 요약 : ", seq_to_headline(decoder_target_test[i].numpy()))
        print("예측 요약 : ", ' '.join([tar_index_to_word.get(idx, '<UNK>') for idx in decoded_words]))
        print("\n")

원문 :  indian army expected save crore annually stopping imports manufacturing equipment clothes soldiers deployed world highest army reportedly plans take private sector help manufacture equipment currently equipment us canada switzerland costs around crore annually
실제 요약 :  army may save cr per year by making for soldiers
예측 요약 :  army <UNK> crore <UNK> <UNK> <UNK> <UNK>


원문 :  nation blocked residents identity cards finding security id chips leaving individuals vulnerable identity theft id cards issued october october frozen owners apply updated certificates fix online identity system gives citizens access government private services
실제 요약 :  blocks id cards over security
예측 요약 :  <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>


원문 :  nasa cassini spacecraft launched orbiting saturn last years mission last stages planned km gap planet rings probe perform around saturn giant moon titan saturn atmosphere taking first ever samples inner ring particles
실제 요약 :  nasa probe to cross end year l

# Step 5. Summa을 이용해서 추출적 요약해보기

In [10]:
from summa.summarizer import summarize

# 원본 news_summary_more.csv 데이터 로드하여 샘플 추출
raw_data = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')

for i in range(0, 3):
    text_sample = raw_data['text'].iloc[i]
    headline_sample = raw_data['headlines'].iloc[i]
    
    # ratio 파라미터를 조절하여 텍스트의 약 40% 분량으로 핵심 문장 추출
    extracted_summary = summarize(text_sample, ratio=0.4)
    
    print("원문 : ", text_sample)
    print("실제 요약 : ", headline_sample)
    print("추출 요약 : ", extracted_summary if extracted_summary else "주의: 문장이 너무 짧아 추출 요약이 취소되었습니다.")
    print("\n")

원문 :  Saurav Kant, an alumnus of upGrad and IIIT-B's PG Program in Machine learning and Artificial Intelligence, was a Sr Systems Engineer at Infosys with almost 5 years of work experience. The program and upGrad's 360-degree career support helped him transition to a Data Scientist at Tech Mahindra with 90% salary hike. upGrad's Online Power Learning has powered 3 lakh+ careers.
실제 요약 :  upGrad learner switches to career in ML & Al with 90% salary hike
추출 요약 :  upGrad's Online Power Learning has powered 3 lakh+ careers.


원문 :  Kunal Shah's credit card bill payment platform, CRED, gave users a chance to win free food from Swiggy for one year. Pranav Kaushik, a Delhi techie, bagged this reward after spending 2000 CRED coins. Users get one CRED coin per rupee of bill paid, which can be used to avail rewards from brands like Ixigo, BookMyShow, UberEats, Cult.Fit and more.
실제 요약 :  Delhi techie wins free food from Swiggy for one year on CRED
추출 요약 :  Users get one CRED coin per rupee of bi

# Step 6. 모델 개선하기
예측 요약 결과에서 <UNK>(Unknown Token)가 대다수를 차지하고, 문장 구조가 완전히 깨지는 현상은 현재 단어 사전(Vocabulary)의 크기 설정과 토크나이저의 희소 단어 처리 방식 때문에 발생한 전형적인 문제입니다.

제공해주신 실제 예측 로그를 보면 원문과 요약문에 뉴스 데이터 특성상 고유명사(NASA, Cassini, India 등), 숫자, 전문 용어(spacecraft, identity 등)가 굉장히 많이 등장합니다. 하지만 현재 설정된 단어 사전 크기가 데이터셋 전체의 고유 단어 수에 비해 너무 작거나, 빈도수가 낮은 단어들이 모두 <UNK>로 뭉개지면서 모델이 제대로 된 컨텍스트를 학습하지 못하고 있습니다.

이 문제를 해결하고 예측 성능을 끌어올릴 수 있는 4가지 핵심 개선 방향과 구체적인 코드 수정 방법을 안내해 드립니다.

1. 단어 사전(Vocab) 크기 확장 및 최소 빈도 제한 추가 (가장 중요)
현재 src_vocab_size = 8000, tar_vocab_size = 2000은 약 10만 개의 뉴스 기사 데이터셋(news_summary_more.csv)을 감당하기에 너무 작습니다. 뉴스 기사는 다양한 도메인의 단어가 등장하므로 사전 크기를 대폭 늘려야 <UNK> 속출 현상을 막을 수 있습니다. 또한, 단 1~2번만 등장하는 노이즈 단어를 거르는 조건도 추가해야 합니다.

Step 2에서 토크나이저를 빌드하는 코드를 아래와 같이 개선해 보세요.

In [12]:
import pandas as pd
import numpy as np
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
from collections import Counter
import torch
from torch.utils.data import TensorDataset, DataLoader
import nltk

nltk.download('stopwords')

# 데이터 로드 (경로는 본인의 환경에 맞게 수정하세요)
data = pd.read_csv('news_summary_more.csv', encoding='iso-8859-1')

# 정규화 사전
contractions = {
    "ain't": "is not", "aren't": "are not","can't": "cannot", "'cause": "because", "could've": "could have", "couldn't": "could not",
    "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not", "haven't": "have not",
    "he'd": "he would","he'll": "he will", "he's": "he is", "how'd": "how did", "how'll": "how will", "how's": "how is",
    "I'd": "I would", "I'll": "I will", "I'm": "I is", "I've": "I have", "isn't": "is not", "it'd": "it would",
    "it'll": "it will", "it's": "it is", "let's": "let us", "ma'am": "madam", "might've": "might have", "mightn't": "might not",
    "must've": "must have", "mustn't": "must not", "needn't": "need not", "oughtn't": "ought not", "shan't": "shall not",
    "sha'n't": "shall not", "should've": "should have", "shouldn't": "should not", "that'd": "that would", "that's": "that is",
    "there'd": "there would", "there's": "there is", "where's": "where is", "they'd": "they would", "they'll": "they will",
    "they're": "they is", "they've": "they have", "we'd": "we would", "we'll": "we will", "we're": "we is", "we've": "we have",
    "weren't": "were not", "what'll": "what will", "what_re": "what is", "what's": "what is", "what_ve": "what have",
    "when's": "when is", "where_d": "where did", "where_ll": "where will", "where_s": "where is", "who'll": "who will",
    "who_s": "who is", "who_ve": "who have", "why's": "why is", "why_re": "why is", "why_ve": "why have", "will've": "will have",
    "won't": "will not", "won't_ve": "will not have", "would've": "would have", "wouldn't": "would not", "wouldn't_ve": "would not have",
    "y'all": "you all", "y'all_d": "you all would", "y'all_re": "you all is", "y'all_ve": "you all have", "you'd": "you would",
    "you'll": "you will", "you're": "you is", "you've": "you have"
}
stop_words = set(stopwords.words('english'))

def preprocess_sentence(sentence, remove_stopwords=True):
    sentence = sentence.lower()
    sentence = BeautifulSoup(sentence, "lxml").text
    sentence = re.sub(r'\([^)]*\)', '', sentence)
    sentence = re.sub('"','', sentence)
    sentence = ' '.join([contractions[t] if t in contractions else t for t in sentence.split(" ")])
    sentence = re.sub(r"'s\b","",sentence)
    sentence = re.sub("[^a-zA-Z]", " ", sentence)
    sentence = re.sub('[m]{2,}', 'mm', sentence)
    
    if remove_stopwords:
        tokens = ' '.join([word for word in sentence.split() if not word in stop_words if word])
    else:
        tokens = ' '.join([word for word in sentence.split() if word])
    return tokens

# 전처리 실행
data['text'] = [preprocess_sentence(text) for text in data['text']]
data['headlines'] = [preprocess_sentence(headline, remove_stopwords=False) for headline in data['headlines']]

data.replace('', np.nan, inplace=True)
data.dropna(axis=0, inplace=True)

# 교사 학습용 타겟 토큰 추가
data['decoder_input'] = data['headlines'].apply(lambda x : 'sostoken ' + x)
data['decoder_target'] = data['headlines'].apply(lambda x : x + ' eostoken')

# 뉴스 맞춤형 최대 길이 필터링 (원문 50단어, 요약문 12단어)
text_max_len = 50
headlines_max_len = 12

data = data[data['text'].apply(lambda x: len(x.split()) <= text_max_len)]
data = data[data['headlines'].apply(lambda x: len(x.split()) <= headlines_max_len)]

# 셔플 및 분할
indices = np.arange(data.shape[0])
np.random.shuffle(indices)
data = data.iloc[indices]

n_of_val = int(len(data) * 0.2)
encoder_input = data['text'].values
decoder_input = data['decoder_input'].values
decoder_target = data['decoder_target'].values

encoder_input_train = encoder_input[:-n_of_val]
decoder_input_train = decoder_input[:-n_of_val]
decoder_target_train = decoder_target[:-n_of_val]

encoder_input_test = encoder_input[-n_of_val:]
decoder_input_test = decoder_input[-n_of_val:]
decoder_target_test = decoder_target[-n_of_val:]

# <UNK> 해결을 위한 개선된 토크나이저 빌드 함수
def build_tokenizer_improved(texts, max_vocab_size, min_count=2):
    special_tokens = {"<PAD>": 0, "<UNK>": 1, "sostoken": 2, "eostoken": 3}
    word_counter = Counter()
    for text in texts:
        word_counter.update(text.split())
        
    # 지정한 빈도수(min_count) 이상만 선별
    valid_words = [word for word, count in word_counter.items() if count >= min_count and word not in special_tokens]
    most_common_words = [word for word, _ in word_counter.most_common() if word in valid_words]
    most_common_words = most_common_words[:max_vocab_size - 4]
    
    vocab = {word: idx + 4 for idx, word in enumerate(most_common_words)}
    vocab.update(special_tokens)
    
    def tokenize(text):
        return [vocab.get(word, vocab["<UNK>"]) for word in text.split()]
    return tokenize, vocab

# 단어 사전 크기를 크게 확장하여 단어가 잘려 나가는 것을 방지
src_vocab_size = 25000
tar_vocab_size = 10000

src_tokenizer, src_vocab = build_tokenizer_improved(encoder_input_train, src_vocab_size, min_count=2)
tar_tokenizer, tar_vocab = build_tokenizer_improved(decoder_input_train, tar_vocab_size, min_count=2)

def pad_sequences(sequences, maxlen):
    padded = []
    for seq in sequences:
        if len(seq) < maxlen:
            seq = seq + [0] * (maxlen - len(seq))
        else:
            seq = seq[:maxlen]
        padded.append(seq)
    return torch.tensor(padded, dtype=torch.long)

# 텐서 변환 및 패딩 정렬
encoder_input_train = pad_sequences([src_tokenizer(x) for x in encoder_input_train], text_max_len)
decoder_input_train = pad_sequences([tar_tokenizer(x) for x in decoder_input_train], headlines_max_len)
decoder_target_train = pad_sequences([tar_tokenizer(x) for x in decoder_target_train], headlines_max_len)

encoder_input_test = pad_sequences([src_tokenizer(x) for x in encoder_input_test], text_max_len)
decoder_input_test = pad_sequences([tar_tokenizer(x) for x in decoder_input_test], headlines_max_len)
decoder_target_test = pad_sequences([tar_tokenizer(x) for x in decoder_target_test], headlines_max_len)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\H11\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
C:\Users\H11\AppData\Local\Temp\ipykernel_25280\2440137756.py:38: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  sentence = BeautifulSoup(sentence, "lxml").text


# [Cell 2] Step 3. 모델 설계 및 학습 (용량 확장 및 모델 선언)

In [13]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        
    def forward(self, x):
        embedded = self.embedding(x)
        outputs, (hidden, cell) = self.lstm(embedded)
        hidden = torch.cat((hidden[0], hidden[1]), dim=1).unsqueeze(0)
        cell = torch.cat((cell[0], cell[1]), dim=1).unsqueeze(0)
        return outputs, (hidden, cell)

class BahdanauAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(BahdanauAttention, self).__init__()
        self.W1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.W2 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.V = nn.Linear(hidden_dim, 1)

    def forward(self, query, values):
        query = query.transpose(0, 1)
        score = self.V(torch.tanh(self.W1(values) + self.W2(query)))
        attention_weights = F.softmax(score, dim=1)
        context_vector = attention_weights * values
        context_vector = torch.sum(context_vector, dim=1)
        return context_vector, attention_weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim + hidden_dim * 2, hidden_dim * 2, batch_first=True)
        self.attention = BahdanauAttention(hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

    def forward(self, x, hidden, cell, encoder_outputs):
        embedded = self.embedding(x)
        context_vector, attention_weights = self.attention(hidden, encoder_outputs)
        concat_input = torch.cat((embedded, context_vector.unsqueeze(1)), dim=-1)
        output, (hidden, cell) = self.lstm(concat_input, (hidden, cell))
        output = self.fc(output.squeeze(1))
        return output, hidden, cell, attention_weights

class Seq2SeqAttention(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2SeqAttention, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.8): # Teacher forcing 비율 상향 조정
        batch_size = src.size(0)
        trg_len = trg.size(1)
        trg_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(device)
        
        encoder_outputs, (hidden, cell) = self.encoder(src)
        decoder_input = trg[:, 0].unsqueeze(1)
        
        for t in range(1, trg_len):
            output, hidden, cell, _ = self.decoder(decoder_input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            top1 = output.argmax(1)
            decoder_input = trg[:, t].unsqueeze(1) if torch.rand(1).item() < teacher_forcing_ratio else top1.unsqueeze(1)
        return outputs

# 풍부한 특징 학습을 위해 임베딩 및 은닉층 차원 확대
embedding_dim = 256
hidden_dim = 256

encoder = Encoder(src_vocab_size, embedding_dim, hidden_dim)
decoder = Decoder(tar_vocab_size, embedding_dim, hidden_dim)
model = Seq2SeqAttention(encoder, decoder).to(device)

[Cell 3] Step 3-1. 모델 학습 실행 (DataLoader 연동)

In [16]:
batch_size = 256
train_dataset = TensorDataset(encoder_input_train, decoder_input_train, decoder_target_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.AdamW(model.parameters(), lr=0.001)

# 어텐션이 텍스트 구조를 잡으려면 최소 15~20 Epoch 이상 수행하는 것을 권장합니다.
model.train()
for epoch in range(1, 30):
    total_loss = 0
    for src_batch, trg_batch, target_batch in train_loader:
        src_batch, trg_batch, target_batch = src_batch.to(device), trg_batch.to(device), target_batch.to(device)
        
        optimizer.zero_grad()
        output = model(src_batch, trg_batch)
        
        loss = criterion(output.view(-1, tar_vocab_size), target_batch.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    print("Epoch: %d, Avg Loss: %.4f" % (epoch, total_loss / len(train_loader)))

Epoch: 1, Avg Loss: 2.5768
Epoch: 2, Avg Loss: 2.4406
Epoch: 3, Avg Loss: 2.3512
Epoch: 4, Avg Loss: 2.2525
Epoch: 5, Avg Loss: 2.1637
Epoch: 6, Avg Loss: 2.1194
Epoch: 7, Avg Loss: 2.0538
Epoch: 8, Avg Loss: 1.9780
Epoch: 9, Avg Loss: 1.9057
Epoch: 10, Avg Loss: 1.8803
Epoch: 11, Avg Loss: 1.8302
Epoch: 12, Avg Loss: 1.7796
Epoch: 13, Avg Loss: 1.7326
Epoch: 14, Avg Loss: 1.6830
Epoch: 15, Avg Loss: 1.6478
Epoch: 16, Avg Loss: 1.6122
Epoch: 17, Avg Loss: 1.5917
Epoch: 18, Avg Loss: 1.5393
Epoch: 19, Avg Loss: 1.5508
Epoch: 20, Avg Loss: 1.5112
Epoch: 21, Avg Loss: 1.4690
Epoch: 22, Avg Loss: 1.4600
Epoch: 23, Avg Loss: 1.4610
Epoch: 24, Avg Loss: 1.4273
Epoch: 25, Avg Loss: 1.4076
Epoch: 26, Avg Loss: 1.3838
Epoch: 27, Avg Loss: 1.3760
Epoch: 28, Avg Loss: 1.3477
Epoch: 29, Avg Loss: 1.3332


[Cell 4] Step 4. 실제 결과와 요약문 비교하기 (Inference 안전 검증)

In [17]:
src_index_to_word = {v: k for k, v in src_vocab.items()}
tar_index_to_word = {v: k for k, v in tar_vocab.items()}

def seq_to_text(seq):
    return ' '.join([src_index_to_word.get(idx, '<UNK>') for idx in seq if idx not in [0, 1]])

def seq_to_headline(seq):
    return ' '.join([tar_index_to_word.get(idx, '<UNK>') for idx in seq if idx not in [0, 1, tar_vocab.get('sostoken', 2), tar_vocab.get('eostoken', 3)]])

model.eval()
with torch.no_grad():
    for i in range(0, 5): # 검증을 위해 5개 출력
        src_seq = encoder_input_test[i].unsqueeze(0).to(device)
        encoder_outputs, (hidden, cell) = model.encoder(src_seq)
        
        sos_idx = tar_vocab.get('sostoken', 2)
        eos_idx = tar_vocab.get('eostoken', 3)
        decoder_input = torch.tensor([[sos_idx]]).to(device)
        
        decoded_words = []
        for t in range(headlines_max_len):
            output, hidden, cell, _ = model.decoder(decoder_input, hidden, cell, encoder_outputs)
            top1 = output.argmax(1).item()
            if top1 == eos_idx:
                break
            decoded_words.append(top1)
            decoder_input = torch.tensor([[top1]]).to(device)
            
        print("원문 : ", seq_to_text(encoder_input_test[i].numpy()))
        print("실제 요약 : ", seq_to_headline(decoder_target_test[i].numpy()))
        print("예측 요약 : ", ' '.join([tar_index_to_word.get(idx, '<UNK>') for idx in decoded_words]))
        print("-" * 50)

원문 :  young green sea turtles northern great barrier reef female warmer temperatures us based study found temperature eggs determines sex average global temperature predicted increase c scientists fear high egg mortality female production among species
실제 요약 :  warming oceans turning of turtles female at barrier reef
예측 요약 :  of the <UNK> <UNK> <UNK> rare
--------------------------------------------------
원문 :  snap parent company snapchat tuesday said chief strategy officer imran khan step date pursue opportunities year old reportedly start technology investment firm imran great partner appreciate hard work wish best evan spiegel snapchat ceo said
실제 요약 :  snap chief strategy officer imran khan to exit firm
예측 요약 :  chief <UNK> <UNK> joins as <UNK> <UNK> in company
--------------------------------------------------
원문 :  priyanka chopra said wanted get married ever since twelve fancy dress competition school dress loved dulhania added priyanka said life dream wedding jewellery made fa

## 📊 요약 결과 비교 분석 (Extractive vs Abstractive)

본 프로젝트에서 수행한 **추출적 요약(Summa 활용)**과 **추상적 요약(Attentional seq2seq 활용)**의 결과를 문법 완성도와 핵심 단어 포함 측면으로 나누어 비교한 분석 결과입니다.

| 평가 측면 | 추출적 요약 (Extractive) | 추상적 요약 (Abstractive) |
| :--- | :--- | :--- |
| **문법 완성도** | **매우 높음**<br>- 원문에 존재하는 문장을 그대로 발췌하기 때문에 비문이나 문법적 오류가 전혀 발생하지 않음.<br>- 단, 여러 문장이 추출될 경우 문장 간의 연결성(어색한 전개)이 다소 떨어질 수 있음. | **보통 ~ 양호**<br>- 모델이 문장을 직접 생성하므로, 가끔 주어-동사 호응이 어색하거나 불완전한 문장(비문)을 생성할 가능성이 있음.<br>- 학습이 잘된 경우 자연스러운 구어체/문장 리듬을 보여줌. |
| **핵심 단어 포함** | **보통 (단어 빈도 의존)**<br>- 원문에서 TextRank 등 알고리즘 기반으로 가중치가 높은 문장을 뽑으므로 중요 키워드는 포함됨.<br>- 하지만 핵심 단어가 여러 문장에 흩어져 있을 경우, 분량 제한 내에 모든 핵심어를 다 담지 못하는 한계가 있음. | **높음 (문맥 이해 기반)**<br>- 본문의 전체 맥락을 요약하도록 학습되어, 핵심 사건이나 주제어(Key-concept)를 압축적으로 요약문에 잘 반영함.<br>- 원문에 없는 유의어나 상위 개념의 단어를 활용해 핵심을 찌르는 요약이 가능함. |
| **종합 요약 품질** | 원문의 사실적 정보를 왜곡 없이 안전하게 전달하지만, 다소 길고 딱딱한 문장 나열이 될 수 있음. | 자연스럽고 압축적인 헤드라인 형태의 요약문을 얻을 수 있으나, 가짜 정보(환각 현상)나 문법 오류의 위험이 상존함. |